## Encode

Fazer one-hot encoding das variáveis categóricas selecionadas, como tnm, loctupri, sexo, etc

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Load directly to Pandas
pdf = pd.read_parquet("/Volumes/workspace/default/cancer_data/01_processed.parquet")
print(f"Loaded {len(pdf)} rows.")

Para um modelo de RSF, os códigos de CID-O não devem ser usados como valores numéricos brutos, pois isso introduz uma ordem artificial sem significado clínico; em vez disso:

- tratar a variável como categórica
- agrupar categorias raras (por exemplo, localizações com baixa frequência) em uma classe “Outros”
- aplicar one-hot encoding 

### NÃO fazer encoding (usar direto)

### Numéricas reais

* `idade`
* `tempo_total_doenca`
* `tempo_ate_consulta`
* `tempo_ate_tratamento`

### Binárias (opcional mapear 1/2 → 0/1)

* `tipo_caso`
* `sexo`
* `historico_familiar_cancer`
* `mais_de_um_tumor_primario`
* `status_vital` (**target**)

---

### Ordinais verdadeiros → **manter código / label encoding**

Aqui **existe progressão clínica real**.

* `escolaridade`
* `t_tnm`
* `n_tnm`
* `m_tnm`
* `t_ptnm`
* `n_ptnm`
* `m_ptnm`
* `comportamento_histologico_tumor`

**Não usar one-hot** (perde informação de progressão).

---

### CATEGÓRICAS NOMINAIS → **FAZER encoding**

### One-Hot Encoding (baixa cardinalidade)

Códigos **sem ordem clínica**

* `raca_cor`
* `uf_procedencia` -> mas agrupar por regioes antes de one-hot
* `uf_hospital` -> mas agrupar por regioes antes de one-hot
* `origem_encaminhamento` 
* `exames_relevantes_diagnostico`
* `diagnostico_tratamento_anteriores`
* `base_diagnostico_mais_importante`
* `base_diagnostico_microscopica`
* `primeiro_tratamento_hospital`
* `razao_nao_tratamento_hospital`
* `cat_localizacao_primaria`
* `subcat_localizacao_primaria`

---

### Alta cardinalidade (muitos valores diferentes) → **Target encoding ou agrupamento**

* `tipo_histologico_tumor` ->  transformar em string e joga categorical encoder ou target encoder
* `ocupacao_principal` -> fazer gap enconder

---

### CASO ESPECIAL (obrigatório recodificar antes do encoding)

`historico_alcoolismo`

`historico_tabagismo`

Códigos **1–9 são rótulos administrativos**, não ordinais.

### Procedimento correto:

1. Criar variável clínica:

   * `nunca`, `ex`, `atual`
2. Criar flag:

   * `info_ausente = 1` se código ∈ {4, 8, 9}
3. One-hot **apenas** da variável clínica

Uma variável vira caso especial quando um único campo mistura:
- Estado clínico real
- Ausência administrativa de informação
- Não aplicabilidade estrutural

Isso exige separação semântica antes do encoding.

---

### TABELA-RESUMO FINAL

| Tipo | Colunas |
|---|---|
| **Numéricas reais** | `idade`, `tempo_total_doenca`, `tempo_ate_consulta`, `tempo_ate_tratamento`, `tempo_ate_obito` |
| **Binárias (1/2 → 0/1)** | `tipo_caso`, `sexo`, `historico_familiar_cancer`, `mais_de_um_tumor_primario`, `status_vital` |
| **Ordinais (label encoding)** | `escolaridade`, `t_tnm`, `n_tnm`, `m_tnm`, `t_ptnm`, `n_ptnm`, `m_ptnm`, `comportamento_histologico_tumor` |
| **One-Hot** | `raca_cor`, `uf_procedencia_regiao`, `uf_hospital_regiao`, `origem_encaminhamento`, `exames_relevantes_diagnostico`, `diagnostico_tratamento_anteriores`, `base_diagnostico_mais_importante`, `base_diagnostico_microscopica`, `primeiro_tratamento_hospital`, `razao_nao_tratamento_hospital`, `cat_localizacao_primaria`*, `historico_tabagismo_clinico`, `historico_alcoolismo_clinico` |
| **Target encoding** | `tipo_histologico_tumor`, `subcat_localizacao_primaria`* |
| **Gap encoding** | `ocupacao_principal` |
| **Flags de ausência** | `historico_tabagismo_info_ausente`, `historico_alcoolismo_info_ausente` |
| **Dropadas** | `historico_tabagismo`, `historico_alcoolismo`, `localizacao_primaria_detalhada`, `uf_procedencia`, `uf_hospital` |

*`cat_localizacao_primaria`: agrupar categorias com < 1% em "outros" antes do OHE
*`subcat_localizacao_primaria`: alta cardinalidade → mover para target encoding



| Coluna | Valores brutos | Descrição | Encoding |
|---|---|---|---|
| `tipo_caso` | `1`=Analítico, `2`=Não analítico | Se o caso foi tratado/acompanhado no hospital | `map_bin`: 1→0, 2→1 |
| `sexo` | `1`=Masculino, `2`=Feminino | Sexo biológico | `map_bin`: 1→0, 2→1 |
| `idade` | numérico (anos) | Idade no diagnóstico | cast `double`, usar direto |
| `raca_cor` | `1`–`5`, `9` | Raça/cor autodeclarada | OHE |
| `escolaridade` | numérico ordinal | Grau de instrução | label encoding (ordinal) |
| `historico_familiar_cancer` | `1`=Não, `2`=Sim → será `0/1` | Familiar de 1º grau com câncer | `map_bin`: 1→0, 2→1 |
| `historico_alcoolismo` | `1`–`3` clínico, `4`,`8`,`9` ausência | Histórico de consumo de álcool | recodificar: `_clinico` (OHE) + `_info_ausente` (flag) |
| `historico_tabagismo` | `1`–`3` clínico, `4`,`8`,`9` ausência | Histórico de tabagismo | recodificar: `_clinico` (OHE) + `_info_ausente` (flag) |
| `uf_procedencia` | sigla UF (ex: `MG`, `SP`) | UF de residência do paciente | agrupar em região → OHE, drop original |
| `origem_encaminhamento` | `1`=SUS, `2`=Não SUS, `3`=Conta própria, `8`, `9` | Como o paciente chegou ao hospital | OHE |
| `exames_relevantes_diagnostico` | `1`–`5`, `8`, `9` | Tipo de exame mais relevante no diagnóstico | OHE |
| `diagnostico_tratamento_anteriores` | categórica | Se houve diagnóstico/tratamento prévio | OHE |
| `base_diagnostico_mais_importante` | `1`–`7`, `9` | Base mais importante usada para fechar diagnóstico | OHE |
| `localizacao_primaria_detalhada` | CID-O (ex: `C44.7`) | Localização anatômica do tumor primário | extrair `cat_localizacao_primaria` + `subcat_localizacao_primaria` (target encoding), drop original |
| `tipo_histologico_tumor` | CID-O morfologia (ex: `8720/2`) | Tipo histológico + comportamento do tumor | extrair `comportamento_histologico_tumor` (ordinal) + `tipo_histologico_tumor` numérico (target encoding) |
| `mais_de_um_tumor_primario` | `1`=Não, `2`=Sim → será `0/1` | Presença de múltiplos tumores primários | `map_bin`: 1→0, 2→1 |
| `razao_nao_tratamento_hospital` | categórica | Motivo pelo qual não foi tratado no hospital | OHE |
| `primeiro_tratamento_hospital` | `1`–`8`, `9` | Primeiro tratamento realizado no hospital | OHE |
| `uf_hospital` | sigla UF (ex: `MG`) | UF do hospital de tratamento | agrupar em região → OHE, drop original |
| `ocupacao_principal` | CBO (ex: `999`, `214`) | Ocupação principal do paciente | gap encoding |
| `base_diagnostico_microscopica` | categórica | Base microscópica do diagnóstico | OHE |
| `t_tnm` | `-1`=X, `0`–`4` | Tamanho/extensão do tumor primário | label encoding (ordinal), usar direto |
| `n_tnm` | `-1`=X, `0`–`3` | Comprometimento de linfonodos regionais | label encoding (ordinal), usar direto |
| `m_tnm` | `-1`=X, `0`–`1` | Presença de metástase à distância | label encoding (ordinal), usar direto |
| `t_ptnm` | `-1`=X, `0`–`4` | Estadiamento patológico — tumor | label encoding (ordinal), usar direto |
| `n_ptnm` | `-1`=X, `0`–`3` | Estadiamento patológico — linfonodos | label encoding (ordinal), usar direto |
| `m_ptnm` | `-1`=X, `0`–`1` | Estadiamento patológico — metástase | label encoding (ordinal), usar direto |
| `status_vital` | `0`=Óbito, `1`=Vivo | Desfecho — **target** | cast `int`, **não entra em** `map_bin` |
| `tempo_total_doenca` | dias (numérico) | Tempo total desde diagnóstico até fim do acompanhamento | cast `double`, usar direto |
| `tempo_ate_consulta` | dias (numérico) | Tempo entre diagnóstico e primeira consulta | cast `double`, usar direto |
| `tempo_ate_tratamento` | dias (numérico) | Tempo entre diagnóstico e início do tratamento | cast `double`, usar direto |
| `tempo_ate_obito` | dias (numérico) ou `null` | Tempo até óbito — `null` para censurados (vivos em 31/12/2025) | cast `double`, usar direto — **não entra nas features do RF**, só no RSF como tempo do evento |
| `cat_localizacao_primaria` | inteiro (ex: `44`, `73`) — extraído do CID-O | Categoria principal da localização (tronco do CID-O) | agrupar raros (<1%) em `"outros"` → OHE |
| `subcat_localizacao_primaria` | inteiro (ex: `7`, `9`) — extraído do CID-O | Subcategoria da localização (dígitos após o ponto) | target encoding |
| `comportamento_histologico_tumor` | `0`–`3` — extraído do CID-O morfologia | Comportamento do tumor (benigno→maligno) | label encoding (ordinal) |


In [0]:
# ____________________________________________________________________________________
# RECODIFICAÇÃO ESPECIAL: historico_tabagismo e historico_alcoolismo
# ____________________________________________________________________________________

def recodificar_habito(pdf, colname):
    # _clinico: 1, 2, 3 -> mantem, resto -> NaN (vai ser tratado no OHE ou fillna)
    pdf[f'{colname}_clinico'] = pdf[colname].where(pdf[colname].isin(['1', '2', '3']))
    # _info_ausente: 1 se for '4', '8', '9' ou nulo, senão 0
    pdf[f'{colname}_info_ausente'] = pdf[colname].isin(['4', '8', '9']).astype(int)
    
    # Preencher nulos nas originais que tbm são ausentes
    pdf.loc[pdf[colname].isna(), f'{colname}_info_ausente'] = 1
    return pdf

for hab in ["historico_tabagismo", "historico_alcoolismo"]:
    if hab in pdf.columns:
        pdf = recodificar_habito(pdf, hab)
        pdf = pdf.drop(columns=[hab])


In [0]:
ordinais_presentes = [c for c in ordinal_cols if c in pdf.columns]
for c in ordinais_presentes:
    pdf[c] = pd.to_numeric(pdf[c], errors='coerce').astype('Int64')


['escolaridade',
 't_tnm',
 'n_tnm',
 'm_tnm',
 't_ptnm',
 'n_ptnm',
 'm_ptnm',
 'comportamento_histologico_tumor']

In [0]:
# ____________________________________________________________________________________
# MAPEAMENTO UF → REGIÃO
# ____________________________________________________________________________________

regioes = {
    'Norte': ['AC', 'AL', 'AP', 'AM', 'PA', 'RO', 'RR'],
    'Nordeste': ['MA', 'PB', 'PI', 'PE', 'RN', 'CE', 'AL', 'SE', 'BA'], # Corrigido: AL repetido e alguns do NE
    'Centro-Oeste': ['GO', 'MT', 'MS', 'DF'],
    'Sudeste': ['ES', 'MG', 'RJ', 'SP'],
    'Sul': ['PR', 'RS', 'SC']
}

# Criar um dict invertido (UF -> Regiao)
uf_to_regiao = {}
for regiao, ufs in regioes.items():
    for uf in ufs:
        uf_to_regiao[uf] = regiao

for uf_col in ["uf_procedencia", "uf_hospital"]:
    if uf_col in pdf.columns:
        pdf[f"{uf_col}_regiao"] = pdf[uf_col].map(uf_to_regiao).fillna('Desconhecida')
        pdf = pdf.drop(columns=[uf_col])


In [0]:
# ____________________________________________________________________________________
# CAST DE TIPOS: numéricas reais → double, binárias → int (0/1)
# Normalização não é necessária para RSF (árvores são invariantes a escala)
# ____________________________________________________________________________________

# Numéricas reais: dias ou anos — cast para double preserva decimais
# tempo_ate_obito é null para censurados (status_vital=1, vivos em 31/12/2025)
# no RSF ele entra como variável de tempo do evento, não como feature
real_cols = [
    "idade",
    "tempo_total_doenca",
    "tempo_ate_consulta",
    "tempo_ate_tratamento",
    "tempo_ate_obito",
]

# ____________________________________________________________________________________
# Binárias com codificação 1/2 → remapear para 0/1
# ATENÇÃO: status_vital já é 0/1 nativo — NÃO entra no map_bin
# map_bin em status_vital converteria: 1(vivo)→0 e 2→1, corrompendo o target
# ____________________________________________________________________________________

binary_cols_para_mapear = [
    "tipo_caso",               # 1=Analítico→0,     2=Não analítico→1
    "sexo",                    # 1=Masculino→0,      2=Feminino→1
    "historico_familiar_cancer",  # 1=Não→0,         2=Sim→1
    "mais_de_um_tumor_primario",  # 1=Não→0,         2=Sim→1
]

def map_bin(colname):
    return (
        F.when(F.col(colname).isin("1", 1), F.lit(0))  # aceita string ou int
         .when(F.col(colname).isin("2", 2), F.lit(1))
         .otherwise(F.col(colname))                     # outros valores passam intactos
    )

for c in real_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

for c in binary_cols_para_mapear:
    if c in df.columns:
        df = df.withColumn(c, map_bin(c).cast("int"))

# status_vital: cast direto para int — já é 0=Óbito, 1=Vivo
if "status_vital" in df.columns:
    df = df.withColumn("status_vital", F.col("status_vital").cast("int"))

# Verificação dos tipos resultantes
all_cols = real_cols + binary_cols_para_mapear + ["status_vital"]
{c: df.schema[c].dataType.simpleString() for c in all_cols if c in df.columns}

{'idade': 'double',
 'tempo_total_doenca': 'double',
 'tempo_ate_consulta': 'double',
 'tempo_ate_tratamento': 'double',
 'tempo_ate_obito': 'double',
 'tipo_caso': 'int',
 'sexo': 'int',
 'historico_familiar_cancer': 'int',
 'mais_de_um_tumor_primario': 'int',
 'status_vital': 'int'}

In [0]:
# Save data locally
import os
os.makedirs("data/interim", exist_ok=True)

train_pdf.to_parquet("/Volumes/workspace/default/cancer_data/03_train.parquet", index=False)
test_pdf.to_parquet("/Volumes/workspace/default/cancer_data/03_test.parquet", index=False)

print(f"Data saved: {len(train_pdf)} train rows, {len(test_pdf)} test rows")


{'train_table': 'data_train', 'test_table': 'data_test', 'train_rows': 58767, 'test_rows': 14692}
